# Rainfall Weather Shock Scenario

This notebook demonstrates the hackathon live-demo behavior: a core weather assumption changes mid-run, and the allocation agent responds with a different decision and different reasoning.

We keep the original cached weather and Sybilion forecast files unchanged. The scenario loads the cached Gedo weather forecast, copies the drought-related weather forecasts in memory, and replaces them with a compound drought shock: collapsing rainfall, falling humidity, and rising temperature. The agent is then evaluated twice for Buur Hakaba:

- baseline cached weather
- modified compound drought-shock weather

The point is not to claim that this rainfall path is the true forecast. It is a controlled stress test: if rainfall expectations collapse while humidity falls and temperature rises, drought pressure should become much more important and the allocation should visibly shift.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in [current, *current.parents]
    if (candidate / "config.json").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import importlib

rainfall_shock_module = importlib.import_module(
    "scenarios.rainfall_weather_shock.run_rainfall_shock"
)
rainfall_shock_module = importlib.reload(rainfall_shock_module)

DEFAULT_OUT_DIR = rainfall_shock_module.DEFAULT_OUT_DIR
run_rainfall_shock_scenario = rainfall_shock_module.run_rainfall_shock_scenario
write_scenario_outputs = rainfall_shock_module.write_scenario_outputs

## Scenario Setup

The shock is controlled by three parameters. `rainfall_end_factor=0.005` forces the final rainfall forecast month down to 0.5% of month one. `humidity_end_factor=0.35` pushes humidity down to 35% of month one. `temperature_increase_c=4.0` raises the final temperature forecast by 4 degrees Celsius. The agent still uses the same water, food, fuel, budget, and deterministic scoring logic; only weather assumptions change.

In [ ]:
REGION = "Buur Hakaba"
BUDGET = 100.0
RAINFALL_END_FACTOR = 0.005
HUMIDITY_END_FACTOR = 0.35
TEMPERATURE_INCREASE_C = 4.0

result = run_rainfall_shock_scenario(
    region=REGION,
    budget=BUDGET,
    rainfall_end_factor=RAINFALL_END_FACTOR,
    humidity_end_factor=HUMIDITY_END_FACTOR,
    temperature_increase_c=TEMPERATURE_INCREASE_C,
)

result.summary

## Rainfall Assumption Shift

The charts below show baseline cached weather forecasts against the modified drought-shock paths used in the stress test.

In [ ]:
weather_shocks = [
    (result.modified_rainfall, "Rainfall forecast stress test", "rainfall mm per day"),
    (result.modified_humidity, "Humidity forecast stress test", "relative humidity (%)"),
    (result.modified_temperature, "Temperature forecast stress test", "average temperature (C)"),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (frame, title, ylabel) in zip(axes, weather_shocks):
    ax.plot(frame["date"], frame["baseline_forecast"], marker="o", label="baseline")
    ax.plot(frame["date"], frame["forecast"], marker="o", label="shock")
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
axes[0].legend()
fig.tight_layout()

## Allocation Change

The table compares the baseline decision to the compound drought-shock decision. The last column shows the percentage-point shift relative to the baseline for each cargo class.

In [ ]:
allocation_table = result.comparison[[
    "scenario",
    "cargo_type",
    "budget_percent",
    "budget_percent_delta_vs_baseline",
    "score",
    "drought_score",
]]
allocation_table

In [ ]:
pivot = result.comparison.pivot(
    index="cargo_type",
    columns="scenario",
    values="budget_percent",
).loc[["water_supplies", "water_infrastructure", "food_supplies", "fuel"]]

ax = pivot.plot(kind="bar", figsize=(9, 4), rot=20)
ax.set_title("Agent allocation before and after rainfall shock")
ax.set_ylabel("budget allocation (%)")
ax.grid(axis="y", alpha=0.3)
ax.legend(title="scenario")
ax.figure.tight_layout()

## Reasoning Change

The reasoning is deterministic and comes from the same score components used for allocation. In the baseline, some allocations are still driven by market-price features. Under the compound weather shock, drought pressure becomes the dominant explanation for water-related decisions and also changes the spillover pressure that affects fuel.

In [ ]:
reasoning_table = result.comparison[[
    "scenario",
    "cargo_type",
    "top_reason",
]]
pd.set_option("display.max_colwidth", 160)
reasoning_table

## Save Scenario Artifacts

The notebook can write the same CSV/JSON artifacts as the CLI script. These files are generated outputs and are ignored by git.

In [ ]:
output_dir = write_scenario_outputs(result, DEFAULT_OUT_DIR)
output_dir